In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ROOT_DIR = '/content/drive/MyDrive/NLP'

Mounted at /content/drive


In [ ]:
import json
import os
import pickle
import time
import numpy as np
import pandas as pd
from datetime import datetime
import random
import torch

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, hinge_loss
)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True
    torch.backends.cudnn.benchmark=False

SEED=42

set_seed(SEED)

sns.set_style("whitegrid")
print("Libraries imported successfully.")

DATASET = "SST-2"
MODEL = "NB-SVM"
C_VALUE = 0.5
N_ITERATIONS = 100
TEST_LABELS = False
SEED = 42

MAX_FEATURES = 200000
NGRAM_RANGE = (1, 3)
SUBLINEAR_TF = True
STRIP_ACCENTS = "unicode"

TEXT_COLUMN = "sentence_preprocessed"
LABEL_COLUMN = "label"
#ROOT_DIR = os.getcwd()

print(f"Dataset:      {DATASET}")
print(f"Max features: {MAX_FEATURES}")
print(f"Ngram range:  {NGRAM_RANGE}")
print(f"Sublinear TF: {SUBLINEAR_TF}")

print(f"Strip accents:{STRIP_ACCENTS}")
print(f"Test labels:  {TEST_LABELS}")
print(f"Seed:         {SEED}")
print(f"Text column:  {TEXT_COLUMN}")
print(f"Label column: {LABEL_COLUMN}")
print(f"Root dir:     {ROOT_DIR}")

Libraries imported successfully.
Dataset:      SST-2
Max features: 200000
Ngram range:  (1, 3)
Sublinear TF: True
Strip accents:unicode
Test labels:  False
Seed:         42
Text column:  sentence_preprocessed
Label column: label
Root dir:     /content/drive/MyDrive/NLP


In [ ]:
from scipy.sparse import csr_matrix

def get_nbsvm_features(X, y):
    # calcul pt raportul de la NB (log countul r)
    def pr(y_i):
        p = X[y == y_i].sum(0)
        return (p + 1) / ((y == y_i).sum() + 1)

    r = np.log(pr(1) / pr(0))
    return csr_matrix(r)

def build_model(model_name, C, max_iter):
    if model_name in ["SVM", "NB-SVM"]:
        model = LinearSVC(C=C, max_iter=max_iter, loss="squared_hinge", random_state=SEED)
        params = {"type": "LinearSVC", "C": C, "max_iter": max_iter, "loss": "squared_hinge"}
        loss_type = "squared_hinge"

    elif model_name == "LogisticRegression":
        from sklearn.linear_model import LogisticRegression
        model = LogisticRegression(C=C, max_iter=max_iter, random_state=SEED)
        params = {"type": "LogisticRegression", "C": C, "max_iter": max_iter, "loss": "log_loss"}
        loss_type = "log_loss"

    else:
        raise ValueError(f"Unknown model: {model_name}")

    return model, params, loss_type

def compute_loss(model, X, y, loss_type):
    if loss_type == "log_loss":
        from sklearn.metrics import log_loss
        return log_loss(y, model.predict_proba(X))

    decision = model.decision_function(X)
    if loss_type == "hinge":
        from sklearn.metrics import hinge_loss
        return hinge_loss(y, decision)
    elif loss_type == "squared_hinge":
        y_signed = np.where(y == 1, 1, -1)
        margins = 1 - y_signed * decision
        return np.mean(np.maximum(0, margins) ** 2)
    else:
        raise ValueError(f"Unsupported loss type: {loss_type}")

print(f"Building model framework ready for: {MODEL}")

Building model framework ready for: NB-SVM


# **GRID SEARCH: Implementarea cu IMDB**


In [ ]:
"""
GRID SEARCH: NB-SVM pe IMDB
==============================================================
"""
import json, os, time, itertools, warnings
import numpy as np
import pandas as pd
from datetime import datetime
import random
from scipy.sparse import csr_matrix
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score

PARAM_GRID = {
    "C":             [0.01, 0.05, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0],
    "ngram_range":   [(1, 2), (1, 3)],
    "max_features":  [50_000, 100_000, 200_000, 300_000],
    "sublinear_tf":  [True, False],
}

MAX_ITER       = 500
SEED           = 42
DATASET        = "IMDB"
TEXT_COLUMN    = "text_preprocessed"
LABEL_COLUMN   = "label"
STRIP_ACCENTS  = "unicode"
ROOT_DIR       = '/content/drive/MyDrive/NLP'
OUTPUT_FILE    = os.path.join(ROOT_DIR, "nbsvm_grid_results_imdb.txt")

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)

def load_data():
    dataset_path = os.path.join(ROOT_DIR, DATASET)
    splits = {}
    for name in ("train", "validation"):
        with open(os.path.join(dataset_path, f"{name}.json"), "r", encoding="utf-8") as f:
            splits[name] = pd.DataFrame(json.load(f))
    return splits["train"], splits["validation"]

def get_nbsvm_features(X, y):
    def pr(y_i):
        p = X[y == y_i].sum(0)
        return (p + 1) / ((y == y_i).sum() + 1)
    r = np.log(pr(1) / pr(0))
    return csr_matrix(r)

set_seed(SEED)
warnings.filterwarnings("ignore")

df_train, df_val = load_data()
y_train = df_train[LABEL_COLUMN].values
y_val   = df_val[LABEL_COLUMN].values

keys   = list(PARAM_GRID.keys())
combos = list(itertools.product(*PARAM_GRID.values()))
print(f"Total combinații de parametri: {len(combos)}\n")

results = []
header = f"{'#':>3}  {'C':>6}  {'ngram':>7}  {'max_feat':>9}  {'sub_tf':>6}  {'train_acc':>9}  {'val_acc':>8}  {'val_f1':>7}  {'time_s':>7}"
sep = "─" * len(header)

with open(OUTPUT_FILE, "w", encoding="utf-8") as log:
    log.write(f"NB-SVM Grid Search IMDB — {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\nDataset: {DATASET}  |  Seed: {SEED}\n\n")
    log.write(header + "\n" + sep + "\n")
    print(header + "\n" + sep)

    for idx, values in enumerate(combos, 1):
        params = dict(zip(keys, values))

        # 1. Vectorizare TF-IDF
        tfidf = TfidfVectorizer(
            max_features=params["max_features"],
            ngram_range=params["ngram_range"],
            sublinear_tf=params["sublinear_tf"],
            strip_accents=STRIP_ACCENTS
        )

        X_train_tfidf = tfidf.fit_transform(df_train[TEXT_COLUMN].fillna("").values)
        X_val_tfidf   = tfidf.transform(df_val[TEXT_COLUMN].fillna("").values)

        t0 = time.time()

        # 2. Extragere si aplicare ponderi NB (r)
        r = get_nbsvm_features(X_train_tfidf, y_train)
        X_train_nb = X_train_tfidf.multiply(r).tocsr()
        X_val_nb   = X_val_tfidf.multiply(r).tocsr()

        # 3. Antrenare
        model = LinearSVC(C=params["C"], max_iter=MAX_ITER, loss="squared_hinge", random_state=SEED)
        model.fit(X_train_nb, y_train)

        elapsed = time.time() - t0

        # 4. Predictii si scoruri:
        train_acc = accuracy_score(y_train, model.predict(X_train_nb))
        val_acc = accuracy_score(y_val, model.predict(X_val_nb))
        val_f1 = f1_score(y_val, model.predict(X_val_nb), average="macro", zero_division=0)

        ngram_str = f"{params['ngram_range'][0]}-{params['ngram_range'][1]}"
        row = f"{idx:3d}  {params['C']:6.2f}  {ngram_str:>7}  {params['max_features']:>9}  {'Y':>6}  {train_acc:9.4f}  {val_acc:8.4f}  {val_f1:7.4f}  {elapsed:7.2f}"

        print(row)
        log.write(row + "\n")

        results.append({
            "C": params["C"],
            "ngram_range": ngram_str,
            "max_features": params["max_features"],
            "val_acc": val_acc
        })

    # Top 10 la final
    log.write(f"\n{'=' * len(header)}\nTOP 10 BY VALIDATION ACCURACY\n{'=' * len(header)}\n")
    for rank, r in enumerate(sorted(results, key=lambda x: x["val_acc"], reverse=True)[:10], 1):
        log.write(f"  #{rank}  C={r['C']:<6}  ngram={r['ngram_range']:<7}  max_feat={r['max_features']:<9}  val_acc={r['val_acc']:.4f}\n")

print(f"\nGrid Search complet! Rezultatele sunt salvate in {OUTPUT_FILE}")

Se încarcă datele IMDB (doar Train și Validation)...
Total combinații de parametri: 128

  #       C    ngram   max_feat  sub_tf  train_acc   val_acc   val_f1   time_s
──────────────────────────────────────────────────────────────────────────────
  1    0.01      1-2      50000       Y     0.8565    0.8440   0.8425     0.72
  2    0.01      1-2      50000       Y     0.8515    0.8400   0.8384     0.56
  3    0.01      1-2     100000       Y     0.8505    0.8390   0.8372     0.81
  4    0.01      1-2     100000       Y     0.8459    0.8356   0.8338     0.63
  5    0.01      1-2     200000       Y     0.8442    0.8352   0.8332     0.77
  6    0.01      1-2     200000       Y     0.8407    0.8328   0.8308     0.92
  7    0.01      1-2     300000       Y     0.8413    0.8340   0.8320     0.93
  8    0.01      1-2     300000       Y     0.8385    0.8316   0.8296     1.04
  9    0.01      1-3      50000       Y     0.8597    0.8472   0.8458     0.49
 10    0.01      1-3      50000       Y   

## Top 10 by Validation Accuracy - IMDB

| Rank | C     | N-gram | Max Features | Sublinear TF | Val Accuracy |
|------|-------|--------|--------------|--------------|--------------|
| #1   | 5.00  | 1–3    | 200000       | Y            | 0.9162       |
| #2   | 5.00  | 1–2    | 200000       | N            | 0.9158       |
| #3   | 5.00  | 1–2    | 300000       | N            | 0.9158       |
| #4   | 5.00  | 1–3    | 300000       | Y            | 0.9156       |
| #5   | 10.00 | 1–3    | 200000       | N            | 0.9156       |
| #6   | 10.00 | 1–3    | 300000       | N            | 0.9156       |
| #7   | 5.00  | 1–3    | 200000       | N            | 0.9154       |
| #8   | 2.00  | 1–3    | 200000       | Y            | 0.9152       |
| #9   | 2.00  | 1–3    | 300000       | Y            | 0.9148       |
| #10  | 5.00  | 1–3    | 300000       | N            | 0.9148       |

# **GRID SEARCH: Implementarea cu SST2**


In [ ]:
"""
Grid Search pentru NB-SVM pe SST-2
=========================================================
"""

import json, os, time, itertools, warnings
import numpy as np
import pandas as pd
from datetime import datetime
import random
from scipy.sparse import csr_matrix

from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score

PARAM_GRID = {
    "C":             [0.01, 0.05, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0],
    "ngram_range":   [(1, 2), (1, 3)],
    "max_features":  [50_000, 100_000, 200_000, 300_000],
    "sublinear_tf":  [True, False],
}

MAX_ITER       = 500
SEED           = 42
DATASET        = "SST-2"
TEXT_COLUMN    = "sentence_preprocessed"
LABEL_COLUMN   = "label"
STRIP_ACCENTS  = "unicode"
ROOT_DIR       = '/content/drive/MyDrive/NLP'
OUTPUT_FILE    = os.path.join(ROOT_DIR, "nbsvm_grid_results_sst2.txt")

# ──────────────────────────────────────────────

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)

def load_data():
    dataset_path = os.path.join(ROOT_DIR, DATASET)
    splits = {}
    for name in ("train", "validation", "test"):
        with open(os.path.join(dataset_path, f"{name}.json"), "r", encoding="utf-8") as f:
            splits[name] = pd.DataFrame(json.load(f))
    return splits["train"], splits["validation"], splits["test"]

def get_nbsvm_features(X, y):
    def pr(y_i):
        p = X[y == y_i].sum(0)
        return (p + 1) / ((y == y_i).sum() + 1)
    r = np.log(pr(1) / pr(0))
    return csr_matrix(r)

def main():
    set_seed(SEED)
    warnings.filterwarnings("ignore")

    print(f"Loading data for {DATASET} …")
    df_train, df_val, df_test = load_data()
    y_train = df_train[LABEL_COLUMN].values
    y_val   = df_val[LABEL_COLUMN].values

    keys   = list(PARAM_GRID.keys())
    combos = list(itertools.product(*PARAM_GRID.values()))
    total  = len(combos)
    print(f"Total parameter combinations: {total}\n")

    results = []
    header = (
        f"{'#':>3}  {'C':>6}  {'ngram':>7}  {'max_feat':>9}  "
        f"{'sub_tf':>6}  {'train_acc':>9}  {'val_acc':>8}  {'val_f1':>7}  {'time_s':>7}"
    )
    sep = "─" * len(header)

    with open(OUTPUT_FILE, "w", encoding="utf-8") as log:
        log.write(f"NB-SVM Grid Search SST-2 — {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        log.write(f"Dataset: {DATASET}  |  Seed: {SEED}\n\n")
        log.write(header + "\n")
        log.write(sep + "\n")
        print(header)
        print(sep)

        for idx, values in enumerate(combos, 1):
            params = dict(zip(keys, values))

            C            = params["C"]
            ngram_range  = params["ngram_range"]
            max_features = params["max_features"]
            sublinear_tf = params["sublinear_tf"]

            tfidf = TfidfVectorizer(
                max_features=max_features,
                ngram_range=ngram_range,
                sublinear_tf=sublinear_tf,
                strip_accents=STRIP_ACCENTS,
            )
            X_train_tfidf = tfidf.fit_transform(df_train[TEXT_COLUMN].fillna("").values)
            X_val_tfidf   = tfidf.transform(df_val[TEXT_COLUMN].fillna("").values)

            t0 = time.time()

            r = get_nbsvm_features(X_train_tfidf, y_train)
            X_train_nb = X_train_tfidf.multiply(r).tocsr()
            X_val_nb   = X_val_tfidf.multiply(r).tocsr()

            model = LinearSVC(C=C, max_iter=MAX_ITER, loss="squared_hinge", random_state=SEED)
            model.fit(X_train_nb, y_train)

            elapsed = time.time() - t0

            y_tr_pred = model.predict(X_train_nb)
            y_vl_pred = model.predict(X_val_nb)

            train_acc  = accuracy_score(y_train, y_tr_pred)
            val_acc    = accuracy_score(y_val, y_vl_pred)
            val_f1     = f1_score(y_val, y_vl_pred, average="macro", zero_division=0)

            ngram_str = f"{ngram_range[0]}-{ngram_range[1]}"
            sub_tf_str = 'Y' if sublinear_tf else 'N'

            row = (
                f"{idx:3d}  {C:6.2f}  {ngram_str:>7}  {max_features:>9}  "
                f"{sub_tf_str:>6}  {train_acc:9.4f}  {val_acc:8.4f}  "
                f"{val_f1:7.4f}  {elapsed:7.2f}"
            )
            print(row)
            log.write(row + "\n")

            results.append({
                "C": C,
                "ngram_range": ngram_str,
                "max_features": max_features,
                "sub_tf": sub_tf_str,
                "train_acc": round(train_acc, 4),
                "val_acc": round(val_acc, 4),
                "val_f1": round(val_f1, 4),
                "time_s": round(elapsed, 2),
            })

        top_10_header = f"\n{'=' * len(header)}\nTOP 10 BY VALIDATION ACCURACY\n{'=' * len(header)}\n"
        log.write(top_10_header)
        print(top_10_header, end="")

        sorted_res = sorted(results, key=lambda x: x["val_acc"], reverse=True)
        for rank, r in enumerate(sorted_res[:10], 1):
            top_row = f"  #{rank:<2}  C={r['C']:<6.2f}  ngram={r['ngram_range']:<7}  max_feat={r['max_features']:<9}  sub_tf={r['sub_tf']:<2}  val_acc={r['val_acc']:.4f}\n"
            log.write(top_row)
            print(top_row, end="")

    print(f"\nDone! Results saved to: {OUTPUT_FILE}")

if __name__ == "__main__":
    main()

Loading data for SST-2 …
Total parameter combinations: 128

  #       C    ngram   max_feat  sub_tf  train_acc   val_acc   val_f1   time_s
──────────────────────────────────────────────────────────────────────────────
  1    0.01      1-2      50000       Y     0.8789    0.8085   0.8071     0.95
  2    0.01      1-2      50000       N     0.8785    0.8073   0.8059     2.75
  3    0.01      1-2     100000       Y     0.8731    0.7959   0.7940     0.51
  4    0.01      1-2     100000       N     0.8727    0.7924   0.7903     0.48
  5    0.01      1-2     200000       Y     0.8731    0.7959   0.7940     0.41
  6    0.01      1-2     200000       N     0.8727    0.7924   0.7903     0.43
  7    0.01      1-2     300000       Y     0.8731    0.7959   0.7940     0.43
  8    0.01      1-2     300000       N     0.8727    0.7924   0.7903     0.37
  9    0.01      1-3      50000       Y     0.8785    0.8085   0.8070     0.57
 10    0.01      1-3      50000       N     0.8783    0.8096   0.8082  

## Top 10 by Validation Accuracy - SST2

| Rank | C     | N-gram | Max Features | Sublinear TF | Val Accuracy |
|------|-------|--------|--------------|--------------|--------------|
| #1   | 0.10  | 1–3    | 100000       | N            | 0.8406       |
| #2   | 0.05  | 1–2    | 50000        | N            | 0.8383       |
| #3   | 0.10  | 1–2    | 50000        | Y            | 0.8383       |
| #4   | 0.10  | 1–2    | 50000        | N            | 0.8383       |
| #5   | 0.10  | 1–3    | 100000       | Y            | 0.8383       |
| #6   | 0.05  | 1–2    | 50000        | Y            | 0.8372       |
| #7   | 0.05  | 1–3    | 100000       | N            | 0.8372       |
| #8   | 0.10  | 1–3    | 50000        | Y            | 0.8372       |
| #9   | 0.05  | 1–3    | 100000       | Y            | 0.8360       |
| #10  | 0.10  | 1–3    | 50000        | N            | 0.8360       |

Top 5 runs pe SST2:

In [ ]:
import json
import os
import pickle
import time
import warnings
import numpy as np
import pandas as pd
from datetime import datetime
import random
import torch
from scipy.sparse import csr_matrix

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
from sklearn.exceptions import ConvergenceWarning

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SEED = 42
set_seed(SEED)
sns.set_style("whitegrid")
warnings.filterwarnings("ignore", category=ConvergenceWarning)
print("Libraries imported successfully.")

ROOT_DIR = '/content/drive/MyDrive/NLP'
DATASET = "SST-2"
MODEL = "NB-SVM"

N_ITERATIONS = 20
SUBLINEAR_TF = True
STRIP_ACCENTS = "unicode"
TEXT_COLUMN = "sentence_preprocessed"
LABEL_COLUMN = "label"
loss_type = "squared_hinge"

# --- TOP 5 MODELE EXTRASE DIN REZULTATE SST-2 ---
TOP_5_MODELS = [
    {"name": "Model1", "C": 0.10, "ngram": (1, 3), "max_feat": 100000, "sub_tf": False},
    {"name": "Model2", "C": 0.05, "ngram": (1, 2), "max_feat": 50000,  "sub_tf": False},
    {"name": "Model3", "C": 0.10, "ngram": (1, 2), "max_feat": 50000,  "sub_tf": True},
    {"name": "Model4", "C": 0.10, "ngram": (1, 2), "max_feat": 50000,  "sub_tf": False},
    {"name": "Model5", "C": 0.10, "ngram": (1, 3), "max_feat": 100000, "sub_tf": True}
]

print(f"\Incarcam setul de date: {DATASET}...")
dataset_path = os.path.join(ROOT_DIR, DATASET)

def load_split(split_name):
    with open(os.path.join(dataset_path, f"{split_name}.json"), "r", encoding="utf-8") as f:
        return pd.DataFrame(json.load(f))

df_train = load_split("train")
df_validation = load_split("validation")
df_test = load_split("test")

X_train_text = df_train[TEXT_COLUMN].fillna("").values
y_train = df_train[LABEL_COLUMN].values
X_val_text = df_validation[TEXT_COLUMN].fillna("").values
y_val = df_validation[LABEL_COLUMN].values
X_test_text = df_test[TEXT_COLUMN].fillna("").values

def get_nbsvm_features(X, y):
    def pr(y_i):
        p = X[y == y_i].sum(0)
        return (p + 1) / ((y == y_i).sum() + 1)
    r = np.log(pr(1) / pr(0))
    return csr_matrix(r)

print("\n" + "="*70)
print(f"TOP 5 MODELE PE SST-2")
print("="*70)

for config in TOP_5_MODELS:
    print(f"\n START: {config['name']} | C={config['C']} | Ngram={config['ngram']} | Max_feat={config['max_feat']}")
    start_time = time.time()

    # 2. VECTORIZARE TF-IDF
    tfidf = TfidfVectorizer(
        max_features=config['max_feat'],
        ngram_range=config['ngram'],
        sublinear_tf=config['sub_tf'],
        strip_accents=STRIP_ACCENTS
    )
    X_train_tfidf = tfidf.fit_transform(X_train_text)
    X_val_tfidf = tfidf.transform(X_val_text)
    X_test_tfidf = tfidf.transform(X_test_text)

    # 3. NB-SVM: CALCULUL PONDERILOR (r)
    r = get_nbsvm_features(X_train_tfidf, y_train)

    X_train_final = X_train_tfidf.multiply(r).tocsr()
    X_val_final = X_val_tfidf.multiply(r).tocsr()
    X_test_final = X_test_tfidf.multiply(r).tocsr()

    # 4. ANTRENARE
    train_losses, val_losses = [], []
    train_accs, val_accs = [], []

    for it in range(1, N_ITERATIONS + 1):
        model = LinearSVC(C=config['C'], max_iter=it, loss="squared_hinge", random_state=SEED)
        model.fit(X_train_final, y_train)

        y_tr_pred = model.predict(X_train_final)
        y_vl_pred = model.predict(X_val_final)

        train_accs.append(accuracy_score(y_train, y_tr_pred))
        val_accs.append(accuracy_score(y_val, y_vl_pred))

        dec_tr = model.decision_function(X_train_final)
        dec_vl = model.decision_function(X_val_final)
        y_tr_signed = np.where(y_train == 1, 1, -1)
        y_vl_signed = np.where(y_val == 1, 1, -1)

        train_losses.append(np.mean(np.maximum(0, 1 - y_tr_signed * dec_tr) ** 2))
        val_losses.append(np.mean(np.maximum(0, 1 - y_vl_signed * dec_vl) ** 2))

        if it % 10 == 0 or it == 1:
            print(f"  Iter {it:3d}/{N_ITERATIONS} | Val Loss: {val_losses[-1]:.4f} | Val Acc: {val_accs[-1]:.4f}")

    train_time = time.time() - start_time
    print(f"  Gata în {train_time:.2f} sec | Final Val Acc: {val_accs[-1]:.4f}")

    #5. GRAFICE
    iters_range = list(range(1, N_ITERATIONS + 1))

    # 5.1 Matricea de confuzie
    val_conf_matrix = confusion_matrix(y_val, y_vl_pred).tolist()
    cm_array = np.array(val_conf_matrix)
    cm_labels = sorted(np.unique(y_val).tolist())
    fig_cm, ax_cm = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm_array, annot=True, fmt="d", cmap="Blues", xticklabels=cm_labels, yticklabels=cm_labels, ax=ax_cm)
    ax_cm.set_xlabel("Predicted")
    ax_cm.set_ylabel("Actual")
    ax_cm.set_title(f"{config['name']} ({DATASET}) - Confusion Matrix", fontweight="bold")
    fig_cm.tight_layout()

    # 5.2 Curbe Accuracy
    fig_acc, ax_acc = plt.subplots(figsize=(9, 5))
    ax_acc.plot(iters_range, train_accs, label="Train Accuracy", linewidth=2)
    ax_acc.plot(iters_range, val_accs, label="Validation Accuracy", linewidth=2)
    ax_acc.set_xlabel("Iteration")
    ax_acc.set_ylabel("Accuracy")
    ax_acc.set_title(f"{config['name']} ({DATASET}) - Accuracy", fontweight="bold")
    ax_acc.legend(loc="lower right")
    fig_acc.tight_layout()

    # 5.3 Curbe Loss
    fig_loss, ax_loss = plt.subplots(figsize=(9, 5))
    ax_loss.plot(iters_range, train_losses, label="Train Loss", linewidth=2)
    ax_loss.plot(iters_range, val_losses, label="Validation Loss", linewidth=2)
    ax_loss.set_xlabel("Iteration")
    ax_loss.set_ylabel(f"Loss ({loss_type})")
    ax_loss.set_title(f"{config['name']} ({DATASET}) - Loss", fontweight="bold")
    ax_loss.legend(loc="upper right")
    fig_loss.tight_layout()

    # 5.4 Predictii Distributie
    counts = [np.sum(y_vl_pred == 0), np.sum(y_vl_pred == 1)]
    fig_dist, ax_dist = plt.subplots(figsize=(6, 5))
    bars = ax_dist.bar(["Negative (0)", "Positive (1)"], counts, color=["#e74c3c", "#2ecc71"], edgecolor="black", width=0.5)
    for bar, count in zip(bars, counts):
        ax_dist.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 10, str(count), ha="center", va="bottom", fontweight="bold")
    ax_dist.set_title(f"{config['name']} ({DATASET}) - Predictions", fontweight="bold")
    fig_dist.tight_layout()

    # 5.5 Top Words (NB-SVM)
    TOP_N = 20
    feature_names = tfidf.get_feature_names_out()
    coefs = np.multiply(model.coef_[0], r.toarray().flatten())

    top_pos_idx = np.argsort(coefs)[-TOP_N:]
    top_neg_idx = np.argsort(coefs)[:TOP_N]

    fig_words, (ax_neg, ax_pos) = plt.subplots(1, 2, figsize=(14, 6))
    ax_neg.barh(feature_names[top_neg_idx], coefs[top_neg_idx], color="#e74c3c")
    ax_neg.set_title(f"Top {TOP_N} Negative Words", fontweight="bold")
    ax_neg.invert_yaxis()
    ax_pos.barh(feature_names[top_pos_idx], coefs[top_pos_idx], color="#2ecc71")
    ax_pos.set_title(f"Top {TOP_N} Positive Words", fontweight="bold")
    fig_words.suptitle(f"{config['name']} ({DATASET}) - Top Words", fontweight="bold", fontsize=14)
    fig_words.tight_layout()

    # ==========================================
    # 6. SALVARE REZULTATE PENTRU MODELUL CURENT
    # ==========================================
    timestamp_str = datetime.now().strftime("%H_%M_%S")
    run_name = f"{config['name']}_C{config['C']}_feat{config['max_feat']}_{timestamp_str}"

    output_dir = os.path.join(ROOT_DIR, MODEL, DATASET, "TOP5_Detailed_Runs", run_name)
    os.makedirs(output_dir, exist_ok=True)

    fig_cm.savefig(os.path.join(output_dir, "confusion_matrix.png"), dpi=150, bbox_inches="tight")
    fig_acc.savefig(os.path.join(output_dir, "accuracy_curves.png"), dpi=150, bbox_inches="tight")
    fig_loss.savefig(os.path.join(output_dir, "loss_curves.png"), dpi=150, bbox_inches="tight")
    fig_dist.savefig(os.path.join(output_dir, "prediction_distribution.png"), dpi=150, bbox_inches="tight")
    fig_words.savefig(os.path.join(output_dir, "top_words.png"), dpi=150, bbox_inches="tight")

    metrics = {
        "model": config['name'],
        "dataset": DATASET,
        "run_name": run_name,
        "model_params": {"type": "LinearSVC_NBSVM", "C": config['C'], "max_iter": N_ITERATIONS},
        "seed": SEED,
        "training": {"training_time_seconds": round(train_time, 4), "train_samples": len(df_train)},
        "vectorization": {"method": "TF-IDF + NB-Weights", "max_features": config['max_feat'], "ngram_range": config['ngram']},
        "train_results": {"accuracy": round(train_accs[-1], 4), "loss": round(train_losses[-1], 4)},
        "validation_results": {
            "accuracy": round(val_accs[-1], 4), "loss": round(val_losses[-1], 4),
            "f1_macro": round(f1_score(y_val, y_vl_pred, average="macro"), 4),
            "confusion_matrix": val_conf_matrix
        }
    }
    with open(os.path.join(output_dir, "metrics.json"), "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2, ensure_ascii=False)

    plt.close('all')

print("\n" + "="*70)
print(f"TOATE CELE 5 MODELE AU FOST ANTRENARE ȘI SALVATE!")
print("="*70)

TOP 5 runs pe IMDB

In [ ]:
import json
import os
import pickle
import time
import warnings
import numpy as np
import pandas as pd
from datetime import datetime
import random
import torch
from scipy.sparse import csr_matrix

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
from sklearn.exceptions import ConvergenceWarning

# --- CONFIGURARE SEED ---
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SEED = 42
set_seed(SEED)
sns.set_style("whitegrid")
warnings.filterwarnings("ignore", category=ConvergenceWarning)

ROOT_DIR = '/content/drive/MyDrive/NLP'
DATASET = "IMDB"
MODEL = "NB-SVM"

N_ITERATIONS = 20
STRIP_ACCENTS = "unicode"
TEXT_COLUMN = "text_preprocessed"
LABEL_COLUMN = "label"
loss_type = "squared_hinge"

TOP_5_MODELS = [
    {"name": "Model1", "C": 5.0,  "ngram": (1, 3), "max_feat": 200000, "sub_tf": True},
    {"name": "Model2", "C": 5.0,  "ngram": (1, 2), "max_feat": 200000, "sub_tf": False},
    {"name": "Model3", "C": 5.0,  "ngram": (1, 2), "max_feat": 300000, "sub_tf": False},
    {"name": "Model4", "C": 5.0,  "ngram": (1, 3), "max_feat": 300000, "sub_tf": True},
    {"name": "Model5", "C": 10.0, "ngram": (1, 3), "max_feat": 200000, "sub_tf": False}
]

def load_split(split_name):
    path = os.path.join(ROOT_DIR, DATASET, f"{split_name}.json")
    with open(path, "r", encoding="utf-8") as f:
        return pd.DataFrame(json.load(f))

def get_nbsvm_features(X, y):
    def pr(y_i):
        p = X[y == y_i].sum(0)
        return (p + 1) / ((y == y_i).sum() + 1)
    r = np.log(pr(1) / pr(0))
    return csr_matrix(r)

def calculate_squared_hinge_loss(X, y_true, model):
    dec = model.decision_function(X)
    y_signed = np.where(y_true == 1, 1, -1)
    return np.mean(np.maximum(0, 1 - y_signed * dec) ** 2)

# 1. incarcarea datelor
print(f" Incarcam setul de date: {DATASET}...")
df_train = load_split("train")
df_val   = load_split("validation")
df_test  = load_split("test")

X_train_text = df_train[TEXT_COLUMN].fillna("").values
y_train      = df_train[LABEL_COLUMN].values
X_val_text   = df_val[TEXT_COLUMN].fillna("").values
y_val        = df_val[LABEL_COLUMN].values
X_test_text  = df_test[TEXT_COLUMN].fillna("").values
y_test       = df_test[LABEL_COLUMN].values

print(f"Date incarcate: Train={len(y_train)}, Val={len(y_val)}, Test={len(y_test)}")

# 2. PROCESARE MODELE
for config in TOP_5_MODELS:
    print(f"\n" + "═"*60)
    print(f"ANTRENARE: {config['name']} | C={config['C']} | Ngram={config['ngram']} | SubTF={config['sub_tf']}")
    print("═"*60)

    start_time = time.time()

    # VECTORIZARE
    tfidf = TfidfVectorizer(
        max_features=config['max_feat'],
        ngram_range=config['ngram'],
        sublinear_tf=config['sub_tf'],
        strip_accents=STRIP_ACCENTS
    )

    X_train_tfidf = tfidf.fit_transform(X_train_text)
    X_val_tfidf   = tfidf.transform(X_val_text)
    X_test_tfidf  = tfidf.transform(X_test_text)

    # NB-WEIGHTS (r)
    r = get_nbsvm_features(X_train_tfidf, y_train)
    X_train_final = X_train_tfidf.multiply(r).tocsr()
    X_val_final   = X_val_tfidf.multiply(r).tocsr()
    X_test_final  = X_test_tfidf.multiply(r).tocsr()

    # MONITORIZARE CURBE
    train_losses, val_losses, test_losses = [], [], []
    train_accs, val_accs, test_accs = [], [], []

    for it in range(1, N_ITERATIONS + 1):
        model = LinearSVC(C=config['C'], max_iter=it, loss="squared_hinge", random_state=SEED)
        model.fit(X_train_final, y_train)

        # predictii
        y_tr_pred = model.predict(X_train_final)
        y_vl_pred = model.predict(X_val_final)
        y_te_pred = model.predict(X_test_final)

        # Scoruri
        train_accs.append(accuracy_score(y_train, y_tr_pred))
        val_accs.append(accuracy_score(y_val, y_vl_pred))
        test_accs.append(accuracy_score(y_test, y_te_pred))

        # Loss
        train_losses.append(calculate_squared_hinge_loss(X_train_final, y_train, model))
        val_losses.append(calculate_squared_hinge_loss(X_val_final, y_val, model))
        test_losses.append(calculate_squared_hinge_loss(X_test_final, y_test, model))

        if it % 5 == 0 or it == 1:
            print(f"  Iter {it:2d}/{N_ITERATIONS} | Val Acc: {val_accs[-1]:.4f} | Test Acc: {test_accs[-1]:.4f}")

    train_time = time.time() - start_time
    iters_range = list(range(1, N_ITERATIONS + 1))

    # 3. GRAFICE
    # 3.1 Accuracy (Train vs Val vs Test)
    fig_acc, ax_acc = plt.subplots(figsize=(8, 5))
    ax_acc.plot(iters_range, train_accs, label="Train Acc")
    ax_acc.plot(iters_range, val_accs, label="Val Acc")
    ax_acc.plot(iters_range, test_accs, label="Test Acc", linestyle='--')
    ax_acc.set_title(f"{config['name']} - Accuracy Curves")
    ax_acc.legend()

    # 3.2 Loss
    fig_loss, ax_loss = plt.subplots(figsize=(8, 5))
    ax_loss.plot(iters_range, train_losses, label="Train Loss")
    ax_loss.plot(iters_range, val_losses, label="Val Loss")
    ax_loss.plot(iters_range, test_losses, label="Test Loss", linestyle='--')
    ax_loss.set_title(f"{config['name']} - Loss Curves ({loss_type})")
    ax_loss.legend()

    # 3.3 Confusion Matrix (pe setul de TEST)
    test_cm = confusion_matrix(y_test, y_te_pred)
    fig_cm, ax_cm = plt.subplots(figsize=(6, 5))
    sns.heatmap(test_cm, annot=True, fmt="d", cmap="Greens")
    ax_cm.set_title(f"{config['name']} - Test Confusion Matrix")

    # 4. SALVARE REZULTATE
    timestamp = datetime.now().strftime("%H%M")
    run_folder = f"{config['name']}_IMDB_{timestamp}"
    output_path = os.path.join(ROOT_DIR, MODEL, DATASET, "TOP5_Detailed_Runs", run_folder)
    os.makedirs(output_path, exist_ok=True)

    fig_acc.savefig(os.path.join(output_path, "accuracy_curves.png"), bbox_inches="tight")
    fig_loss.savefig(os.path.join(output_path, "loss_curves.png"), bbox_inches="tight")
    fig_cm.savefig(os.path.join(output_path, "test_confusion_matrix.png"), bbox_inches="tight")

    # JSON METRICS
    final_metrics = {
        "config": config,
        "execution_time": round(train_time, 2),
        "final_train_acc": round(train_accs[-1], 4),
        "final_val_acc": round(val_accs[-1], 4),
        "final_test_acc": round(test_accs[-1], 4),
        "test_f1_macro": round(f1_score(y_test, y_te_pred, average="macro"), 4),
        "classification_report_test": classification_report(y_test, y_te_pred, output_dict=True)
    }

    with open(os.path.join(output_path, "metrics.json"), "w") as f:
        json.dump(final_metrics, f, indent=4)

    plt.close('all')
    print(f" Finalizat: Test Acc = {test_accs[-1]:.4f} | Salva în {run_folder}")

print("\n Toate runurile pentru IMDB au fost finalizate!")

Pentru a genera dummy folder pentru a trimite in GLUE:

In [ ]:
import pandas as pd
import zipfile
import os
import shutil

submission_dir = os.path.join(ROOT_DIR, "GLUE_Submission")
zip_path = os.path.join(ROOT_DIR, "GLUE_SST2_FINAL.zip")

if os.path.exists(submission_dir):
    shutil.rmtree(submission_dir)
if os.path.exists(zip_path):
    os.remove(zip_path)

os.makedirs(submission_dir, exist_ok=True)

y_test_pred_final = model.predict(X_test_final)
sst2_path = os.path.join(submission_dir, "SST-2.tsv")
pd.DataFrame({
    'index': range(len(y_test_pred_final)),
    'prediction': y_test_pred_final
}).to_csv(sst2_path, sep='\t', index=False)
print(" Predictiile pentru SST-2 au fost salvate.")

# format GLUE pentru TOATE SETURILE DE DATE, si pt SST2 este deja pus corect
dummy_tasks_info = {
    "CoLA": {"length": 1063, "label": 0},
    "MRPC": {"length": 1725, "label": 0},
    "QQP": {"length": 390965, "label": 0},
    "STS-B": {"length": 1379, "label": 0.0},
    "MNLI-m": {"length": 9796, "label": "entailment"},
    "MNLI-mm": {"length": 9847, "label": "entailment"},
    "QNLI": {"length": 5463, "label": "entailment"},
    "RTE": {"length": 3000, "label": "entailment"},
    "WNLI": {"length": 146, "label": 0},
    "AX": {"length": 1104, "label": "entailment"}
}

for task, info in dummy_tasks_info.items():
    dummy_path = os.path.join(submission_dir, f"{task}.tsv")
    pd.DataFrame({
        'index': range(info["length"]),
        'prediction': [info["label"]] * info["length"]
    }).to_csv(dummy_path, sep='\t', index=False)


with zipfile.ZipFile(zip_path, 'w') as zipf:
    for file in os.listdir(submission_dir):
        file_path = os.path.join(submission_dir, file)
        zipf.write(file_path, arcname=file)
